In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString
import os

# Cargar el Excel descargado
df = pd.read_excel("data/listado_iiee.xlsx")

# Revisa qué columnas hay
print(df.columns)

In [ ]:
niveles = df['Nivel / Modalidad'].dropna().unique()
for nivel in sorted(niveles):
    print(nivel)

# Convertimos a minúsculas para evitar problemas con mayúsculas/minúsculas
df['nivel_lower'] = df['Nivel / Modalidad'].str.lower()

# Filtros más controlados
filtro_inicial = df['nivel_lower'].str.contains('inicial')
filtro_primaria = df['nivel_lower'].str.contains('primaria')
filtro_secundaria = df['nivel_lower'].str.contains('secundaria')

# Aplicar filtros
df_inicial = df[filtro_inicial]
df_primaria = df[filtro_primaria]
df_secundaria = df[filtro_secundaria]

In [ ]:
# Revisa cuantas escuelas hay por nivel
print(f"Inicial: {len(df_inicial)} escuelas")
print(f"Primaria: {len(df_primaria)} escuelas")
print(f"Secundaria: {len(df_secundaria)} escuelas")

# Revisa cuantas escuelas hay por nivel dentro de inicial y el tipo de dato
print(df_inicial['Nivel / Modalidad'].value_counts())
print(df_primaria['Nivel / Modalidad'].value_counts())
print(df_secundaria['Nivel / Modalidad'].value_counts())

In [ ]:
inicial_count = df_inicial.groupby(['Departamento', 'Provincia', 'Distrito']).size().reset_index(name='n_inicial')
primaria_count = df_primaria.groupby(['Departamento', 'Provincia', 'Distrito']).size().reset_index(name='n_primaria')
secundaria_count = df_secundaria.groupby(['Departamento', 'Provincia', 'Distrito']).size().reset_index(name='n_secundaria')

print(inicial_count.head())
print(primaria_count.head())
print(secundaria_count.head())

In [ ]:
# Cargar shapefile de distritos
gdf_distritos = gpd.read_file("data\shape_file\DISTRITOS.shp")

# Revisa los nombres reales de las columnas en gdf_distritos
print(gdf_distritos.columns)

# Mostrar las primeras filas del GeoDataFrame
print(gdf_distritos.head())

In [ ]:
# Unir la geometría de los distritos con los datos de las escuelas por nivel reemplazando NaN por 0
gdf_inicial = gdf_distritos.merge(inicial_count, left_on=['DEPARTAMEN', 'PROVINCIA', 'DISTRITO'], right_on=['Departamento', 'Provincia', 'Distrito'], how='left').fillna({'n_inicial': 0})
gdf_primaria = gdf_distritos.merge(primaria_count, left_on=['DEPARTAMEN', 'PROVINCIA', 'DISTRITO'], right_on=['Departamento', 'Provincia', 'Distrito'], how='left').fillna({'n_primaria': 0})
gdf_secundaria = gdf_distritos.merge(secundaria_count, left_on=['DEPARTAMEN', 'PROVINCIA', 'DISTRITO'], right_on=['Departamento', 'Provincia', 'Distrito'], how='left').fillna({'n_secundaria': 0})

In [ ]:
missing = gdf_inicial[gdf_inicial['n_inicial'].isna()]
print(missing[['Departamento', 'Provincia', 'Distrito']])

In [ ]:
# Crear el mapa para Inicial
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_inicial.plot(column='n_inicial', cmap='Reds', ax=ax, legend=True,
                 edgecolor='black',
                 legend_kwds={'label': "Cantidad de Escuelas Iniciales"}, 
                 linewidth=0.5)
ax.set_title('Distribución de Escuelas Iniciales por Distrito en Perú')

# Crear el mapa para Primaria
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_primaria.plot(column='n_primaria', cmap='Reds', ax=ax, legend=True,
                 edgecolor='black',
                 legend_kwds={'label': "Cantidad de Escuelas Primarias"}, 
                 linewidth=0.5)
ax.set_title('Distribución de Escuelas Primarias por Distrito en Perú')

# Crear el mapa para Secundaria
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_secundaria.plot(column='n_secundaria', cmap='Reds', ax=ax, legend=True,
                 edgecolor='black',
                 legend_kwds={'label': "Cantidad de Escuelas Secundarias"}, 
                 linewidth=0.5)
ax.set_title('Distribución de Escuelas Secundaria por Distrito en Perú')

# Mostrar los mapas
plt.show()
